In [1]:
import pandas as pd

df = pd.read_csv("IMDB Dataset.csv")
print(df.head())
print(df.columns)

                                              review sentiment
0  One of the other reviewers has mentioned that ...  positive
1  A wonderful little production. <br /><br />The...  positive
2  I thought this was a wonderful way to spend ti...  positive
3  Basically there's a family where a little boy ...  negative
4  Petter Mattei's "Love in the Time of Money" is...  positive
Index(['review', 'sentiment'], dtype='str')


In [26]:
df.shape

(50000, 2)

In [27]:
df = df.head(1000)

In [28]:
df.shape

(1000, 2)

In [29]:
df["label"] = df["sentiment"].map(
    {
        "positive":1,
        "negative":0
    }
)

In [30]:
df.head()

,review,sentiment,label
0,One of the other reviewers has mentioned that ...,positive,1
1,A wonderful little production. <br /><br />The...,positive,1
2,I thought this was a wonderful way to spend ti...,positive,1
3,Basically there's a family where a little boy ...,negative,0
4,"Petter Mattei's ""Love in the Time of Money"" is...",positive,1


In [31]:
from sklearn.model_selection import train_test_split


train_df, test_df = train_test_split(
    df,
    test_size=0.2,
    random_state=42
)

# Create PyTorch Dataset

In [32]:
import torch
from torch.utils.data import Dataset


class IMDBDataset(Dataset):

    def __init__(
        self,
        dataframe,
        tokenizer,
        max_length=256
    ):

        self.texts = dataframe["review"].values
        self.labels = dataframe["label"].values

        self.tokenizer = tokenizer
        self.max_length = max_length



    def __len__(self):

        return len(self.texts)



    def __getitem__(self,index):

        text = self.texts[index]
        label = self.labels[index]


        encoded = self.tokenizer(
            text,
            padding="max_length",
            truncation=True,
            max_length=self.max_length,
            return_tensors="pt"
        )


        return {

            "input_ids":
            encoded["input_ids"].squeeze(0),


            "attention_mask":
            encoded["attention_mask"].squeeze(0),


            "labels":
            torch.tensor(
                label,
                dtype=torch.long
            )

        }

# Load Tokenizer

In [33]:
from transformers import AutoTokenizer


tokenizer = AutoTokenizer.from_pretrained(
    "distilbert-base-uncased"
)

# Create Dataset Objects

In [34]:
train_dataset = IMDBDataset(
    train_df,
    tokenizer
)


test_dataset = IMDBDataset(
    test_df,
    tokenizer
)

In [35]:
sample = train_dataset[0]
print(sample)

{'input_ids': tensor([  101,  1005,  2162,  3185,  1005,  2003,  1037,  5365,  6907,  2008,
         2038,  2042,  2589,  1998,  2417,  5643,  2061,  2116,  2335,  2008,
        18856, 17322,  2094,  7982,  1010,  2128, 14949,  9072,  5436,  1998,
         2058,  1011,  1996,  1011,  2327,  2895, 10071,  4025, 14477,  6767,
         8524,  3468,  2005,  2151,  4736,  7149,  2007,  2312,  1011,  4094,
         4337,  1012,  2320,  1999,  1037,  2096,  1010,  2174,  1010,  1037,
         2162,  3185,  3310,  2247,  2008,  3632,  2114,  1996,  8982,  1998,
         7545,  1037,  5621,  2434,  1998, 17075,  2466,  2000,  2166,  2006,
         1996,  3165,  3898,  1012,  1996,  2942,  2162,  1011,  3690,  1000,
         3147,  3137,  1010,  1000,  4626, 12582,  2375,  1010,  9851,  4845,
         2386,  1998, 17400, 27838,  3363,  8545,  4590,  2003,  2107,  1037,
         2143,  1012,  1026,  7987,  1013,  1028,  1026,  7987,  1013,  1028,
         2059,  2153,  1010,  4214,  3147,  3137, 

In [36]:
from torch.utils.data import DataLoader


train_loader = DataLoader(
    train_dataset,
    batch_size=16,
    shuffle=True
)


test_loader = DataLoader(
    test_dataset,
    batch_size=16
)

In [37]:
batch = next(iter(train_loader))


print(batch["input_ids"].shape)

torch.Size([16, 256])


In [38]:
from transformers import AutoModelForSequenceClassification


model = AutoModelForSequenceClassification.from_pretrained(
    "distilbert-base-uncased",
    num_labels=2
)

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
pre_classifier.bias     | MISSING    | 
classifier.weight       | MISSING    | 
classifier.bias         | MISSING    | 
pre_classifier.weight   | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [39]:
from torch.optim import AdamW


optimizer = AdamW(
    model.parameters(),
    lr=2e-5
)

In [40]:
device = (
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)


model.to(device)

DistilBertForSequenceClassification(
  (distilbert): DistilBertModel(
    (embeddings): Embeddings(
      (word_embeddings): Embedding(30522, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (transformer): Transformer(
      (layer): ModuleList(
        (0-5): 6 x TransformerBlock(
          (attention): DistilBertSelfAttention(
            (q_lin): Linear(in_features=768, out_features=768, bias=True)
            (k_lin): Linear(in_features=768, out_features=768, bias=True)
            (v_lin): Linear(in_features=768, out_features=768, bias=True)
            (out_lin): Linear(in_features=768, out_features=768, bias=True)
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (sa_layer_norm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
          (ffn): FFN(
            (dropout): Dropout(p=0.1, inplace=False)


In [41]:
epochs = 3
for epoch in range(epochs):

    model.train()

    total_loss = 0

    for batch in train_loader:
        input_ids = batch["input_ids"].to(device)

        attention_mask = batch["attention_mask"].to(device)

        labels = batch["labels"].to(device)

        # clear gradients
        optimizer.zero_grad()

        # forward pass
        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            labels=labels
        )

        # loss
        loss = outputs.loss

        # backward propagation
        loss.backward()

        # update weights
        optimizer.step()

        total_loss += loss.item()

    print(
        "Epoch:",
        epoch+1,
        "Loss:",
        total_loss/len(train_loader)
    )

Epoch: 1 Loss: 0.5840276539325714
Epoch: 2 Loss: 0.2815831335633993
Epoch: 3 Loss: 0.11627975471317768


# Validation

In [42]:
from sklearn.metrics import accuracy_score

model.eval()

predictions = []
true_labels = []

with torch.no_grad():

    for batch in test_loader:

        input_ids = batch["input_ids"].to(device)

        attention_mask = batch["attention_mask"].to(device)

        labels = batch["labels"].to(device)

        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask
        )

        logits = outputs.logits


        preds = torch.argmax(
            logits,
            dim=1
        )


        predictions.extend(
            preds.cpu().numpy()
        )

        true_labels.extend(
            labels.cpu().numpy()
        )

accuracy = accuracy_score(
    true_labels,
    predictions
)

print(
    "Accuracy:",
    accuracy
)

Accuracy: 0.925


# Save model 

In [43]:
model.save_pretrained(
    "imdb_sentiment_model"
)

tokenizer.save_pretrained(
    "imdb_sentiment_model"
)

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('imdb_sentiment_model\\tokenizer_config.json',
 'imdb_sentiment_model\\tokenizer.json')

# Inference

In [44]:
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification

device = "cuda" if torch.cuda.is_available() else "cpu"

model_path = "imdb_sentiment_model"


tokenizer = AutoTokenizer.from_pretrained(
    model_path
)

model = AutoModelForSequenceClassification.from_pretrained(
    model_path
)

model.to(device)

model.eval()

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

DistilBertForSequenceClassification(
  (distilbert): DistilBertModel(
    (embeddings): Embeddings(
      (word_embeddings): Embedding(30522, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (transformer): Transformer(
      (layer): ModuleList(
        (0-5): 6 x TransformerBlock(
          (attention): DistilBertSelfAttention(
            (q_lin): Linear(in_features=768, out_features=768, bias=True)
            (k_lin): Linear(in_features=768, out_features=768, bias=True)
            (v_lin): Linear(in_features=768, out_features=768, bias=True)
            (out_lin): Linear(in_features=768, out_features=768, bias=True)
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (sa_layer_norm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
          (ffn): FFN(
            (dropout): Dropout(p=0.1, inplace=False)


In [45]:
def predict_sentiment(text):

    # Tokenize input text
    inputs = tokenizer(
        text,
        padding="max_length",
        truncation=True,
        max_length=256,
        return_tensors="pt"
    )

    # Move tensors to GPU/CPU
    inputs = {
        k:v.to(device)
        for k,v in inputs.items()
    }


    # Disable gradients
    with torch.no_grad():

        outputs = model(**inputs)


    # Get logits
    logits = outputs.logits

    # Convert logits to probabilities
    probabilities = torch.softmax(
        logits,
        dim=1
    )


    # Get class
    prediction = torch.argmax(
        probabilities,
        dim=1
    ).item()


    confidence = probabilities[0][prediction].item()
    label = (
        "positive"
        if prediction == 1
        else "negative"
    )


    return {
        "sentiment": label,
        "confidence": confidence
    }

In [46]:
review = """
This movie was absolutely amazing.
The acting was brilliant and the story was very touching.
"""


result = predict_sentiment(review)

print(result)

{'sentiment': 'positive', 'confidence': 0.9870930314064026}


In [47]:
review = """
This was one of the worst movies I have ever watched.
The plot was boring and the acting was terrible.
"""


print(
    predict_sentiment(review)
)

{'sentiment': 'negative', 'confidence': 0.9842771291732788}
